In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/PakWheels Dataset.csv")

print("Original dataset shape:", df.shape)
df.head()

Original dataset shape: (61919, 15)


,nam,Price,Year,Millage,Fuel,Transmission,Province,Color,Assembly,Engine Capacity,Body Type,Ad Reference,Features,Owner nam,url
0,Suzuki Alto VXL AGS 2022,PKR 23.9 lacs,2022,"120,000 km",Petrol,Automatic,Islamabad,Solid White,Local,660 cc,Hatchback,10313920,"['ABS', 'AM/FM Radio', 'Air Bags', 'Air Condit...",Daniyal,https://www.pakwheels.com//used-cars/suzuki-al...
1,Honda N Box Custom GL 2022,PKR 33.4 lacs,2022,"37,110 km",Petrol,Automatic,Islamabad,White,Imported,658 cc,Hatchback,10233745,"['ABS', 'AM/FM Radio', 'Air Bags', 'Air Condit...",NaN,https://www.pakwheels.com//used-cars/honda-n-b...
2,Honda Civic EX 1995,PKR 6.3 lacs,1995,786 km,LPG,Manual,Karachi,Black,Local,1500 cc,Sedan,10313884,"['AM/FM Radio', 'Air Conditioning', 'Cassette ...",Abdullah Sheikh,https://www.pakwheels.com//used-cars/honda-civ...
3,Toyota Corolla GLi Automatic 1.6 VVTi 2012,PKR 33.25 lacs,2012,"133,000 km",Petrol,Automatic,Lahore,Medium Silver,Imported,1600 cc,Sedan,10311918,"['ABS', 'AM/FM Radio', 'Air Conditioning', 'CD...",Zakiulhassan,https://www.pakwheels.com//used-cars/toyota-co...
4,Toyota Corolla Hatchback 1998,PKR 16.45 lacs,1998,"125,225 km",Petrol,Automatic,Punjab,Black,Local,1600 cc,NaN,10207521,"['ABS', 'AM/FM Radio', 'Air Bags', 'Air Condit...",NaN,https://www.pakwheels.com//used-cars/toyota-co...


In [2]:
df = df.rename(columns={
    "nam": "name",
    "Price": "price",
    "Year": "year",
    "Millage": "mileage",
    "Fuel": "fuel",
    "Transmission": "transmission",
    "Province": "registration_location",
    "Color": "color",
    "Assembly": "assembly",
    "Engine Capacity": "engine_capacity",
    "Body Type": "body_type",
    "Ad Reference": "ad_reference",
    "Features": "features",
    "Owner nam": "owner_name"
})

df.columns.tolist()

['name',
 'price',
 'year',
 'mileage',
 'fuel',
 'transmission',
 'registration_location',
 'color',
 'assembly',
 'engine_capacity',
 'body_type',
 'ad_reference',
 'features',
 'owner_name',
 'url']

In [3]:
def price_to_pkr(price):
    price = str(price).lower().replace("pkr", "").strip()

    if price == "call for price":
        return np.nan

    if "crore" in price:
        return float(price.replace("crore", "").strip()) * 10_000_000

    if "lacs" in price:
        return float(price.replace("lacs", "").strip()) * 100_000

    return np.nan


df["price_pkr"] = df["price"].apply(price_to_pkr)

# Remove rows where the target price is unavailable
df = df.dropna(subset=["price_pkr"]).copy()

print("Shape after removing 'Call for price':", df.shape)
print("Missing price_pkr:", df["price_pkr"].isna().sum())

Shape after removing 'Call for price': (61066, 16)
Missing price_pkr: 0


In [4]:
df["mileage_km"] = pd.to_numeric(
    df["mileage"]
    .str.replace(",", "", regex=False)
    .str.replace("km", "", case=False)
    .str.strip(),
    errors="coerce"
)

print("Missing mileage_km:", df["mileage_km"].isna().sum())
df[["mileage", "mileage_km"]].head(10)

Missing mileage_km: 0


,mileage,mileage_km
0,"120,000 km",120000
1,"37,110 km",37110
2,786 km,786
3,"133,000 km",133000
4,"125,225 km",125225
5,"4,100 km",4100
6,"4,917 km",4917
7,"75,000 km",75000
8,"22,000 km",22000
9,"180,000 km",180000


In [5]:
# Combustion / hybrid engine displacement
df["engine_cc"] = np.where(
    df["engine_capacity"].str.contains("cc", case=False, na=False),
    pd.to_numeric(
        df["engine_capacity"]
        .str.replace("cc", "", case=False)
        .str.strip(),
        errors="coerce"
    ),
    np.nan
)

# Electric vehicle battery capacity
df["battery_kwh"] = np.where(
    df["engine_capacity"].str.contains("kWh", case=False, na=False),
    pd.to_numeric(
        df["engine_capacity"]
        .str.replace("kWh", "", case=False)
        .str.strip(),
        errors="coerce"
    ),
    np.nan
)

print("engine_cc values:", df["engine_cc"].notna().sum())
print("battery_kwh values:", df["battery_kwh"].notna().sum())

df[["fuel", "engine_capacity", "engine_cc", "battery_kwh"]].sample(10)

engine_cc values: 60595
battery_kwh values: 471


,fuel,engine_capacity,engine_cc,battery_kwh
31937,Petrol,660 cc,660.0,NaN
53054,Petrol,800 cc,800.0,NaN
38983,Petrol,1000 cc,1000.0,NaN
10443,Petrol,1800 cc,1800.0,NaN
24134,Petrol,1500 cc,1500.0,NaN
27662,Petrol,1200 cc,1200.0,NaN
18802,Hybrid,1500 cc,1500.0,NaN
52040,Petrol,1300 cc,1300.0,NaN
35320,Petrol,1000 cc,1000.0,NaN
18383,Diesel,2800 cc,2800.0,NaN


In [6]:
print("Rows with year 1900 before cleaning:", (df["year"] == 1900).sum())

df = df[df["year"] != 1900].copy()

print("Dataset shape after removing invalid year:", df.shape)
print("Minimum year now:", df["year"].min())

Rows with year 1900 before cleaning: 2
Dataset shape after removing invalid year: (61064, 19)
Minimum year now: 1952


In [7]:
print(
    "Duplicate ad references before cleaning:",
    df["ad_reference"].duplicated().sum()
)

df = df.drop_duplicates(
    subset="ad_reference",
    keep="last"
).copy()

print(
    "Duplicate ad references after cleaning:",
    df["ad_reference"].duplicated().sum()
)

print("Dataset shape:", df.shape)

Duplicate ad references before cleaning: 8
Duplicate ad references after cleaning: 0
Dataset shape: (61056, 19)


In [8]:
invalid_engine_cc = (
    (df["engine_cc"] < 500) |
    (df["engine_cc"] > 7000)
)

print("Suspicious engine_cc values:", invalid_engine_cc.sum())

df.loc[invalid_engine_cc, "engine_cc"] = np.nan

print(
    "Suspicious engine_cc values remaining:",
    ((df["engine_cc"] < 500) | (df["engine_cc"] > 7000)).sum()
)

print("Missing engine_cc after cleaning:", df["engine_cc"].isna().sum())

Suspicious engine_cc values: 66
Suspicious engine_cc values remaining: 0
Missing engine_cc after cleaning: 537


In [9]:
(df["battery_kwh"] > 500).sum()

np.int64(36)

In [10]:
invalid_battery = df["battery_kwh"] > 500

print("Suspicious battery values:", invalid_battery.sum())

df.loc[invalid_battery, "battery_kwh"] = np.nan

print(
    "Battery values above 500 remaining:",
    (df["battery_kwh"] > 500).sum()
)

print(
    "Missing battery_kwh:",
    df["battery_kwh"].isna().sum()
)

Suspicious battery values: 36
Battery values above 500 remaining: 0
Missing battery_kwh: 60621


In [11]:
print("Mileage = 1 km:", (df["mileage_km"] == 1).sum())
print("Mileage = 1,000,000 km:", (df["mileage_km"] == 1_000_000).sum())

Mileage = 1 km: 457
Mileage = 1,000,000 km: 59


In [12]:
mileage_placeholder = (
    (df["mileage_km"] == 1) |
    (df["mileage_km"] == 1_000_000)
)

print("Mileage placeholders:", mileage_placeholder.sum())

df.loc[mileage_placeholder, "mileage_km"] = np.nan

print("Missing mileage_km after cleaning:", df["mileage_km"].isna().sum())

Mileage placeholders: 516
Missing mileage_km after cleaning: 516


In [13]:
df = df.drop(
    columns=["owner_name", "url", "ad_reference"]
)

print("Dataset shape:", df.shape)
print(df.columns.tolist())

Dataset shape: (61056, 16)
['name', 'price', 'year', 'mileage', 'fuel', 'transmission', 'registration_location', 'color', 'assembly', 'engine_capacity', 'body_type', 'features', 'price_pkr', 'mileage_km', 'engine_cc', 'battery_kwh']


In [14]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
})

missing_summary = missing_summary[
    missing_summary["missing_count"] > 0
].sort_values("missing_count", ascending=False)

missing_summary

,missing_count,missing_percent
battery_kwh,60621,99.29
body_type,7115,11.65
features,3852,6.31
engine_cc,537,0.88
mileage_km,516,0.85
color,1,0.00


In [15]:
df["color"] = df["color"].fillna("Unknown")

print("Missing color values:", df["color"].isna().sum())
print("Unknown color values:", (df["color"] == "Unknown").sum())

Missing color values: 0
Unknown color values: 1


In [16]:
df["body_type"] = df["body_type"].fillna("Unknown")

print("Missing body_type:", df["body_type"].isna().sum())
print("Unknown body_type:", (df["body_type"] == "Unknown").sum())

Missing body_type: 0
Unknown body_type: 7115


In [17]:
import ast

def count_features(value):
    if pd.isna(value):
        return 0

    try:
        return len(ast.literal_eval(value))
    except (ValueError, SyntaxError, TypeError):
        return 0

df["feature_count"] = df["features"].apply(count_features)

print(df["feature_count"].describe())
print("Missing feature_count:", df["feature_count"].isna().sum())

count    61056.000000
mean         9.598680
std          5.170618
min          0.000000
25%          5.000000
50%         11.000000
75%         13.000000
max         28.000000
Name: feature_count, dtype: float64
Missing feature_count: 0


In [18]:
non_missing_features = df["features"].notna()

parse_failures = (
    non_missing_features &
    (df["feature_count"] == 0)
)

print("Non-missing feature rows:", non_missing_features.sum())
print("Potential parsing failures:", parse_failures.sum())

Non-missing feature rows: 57204
Potential parsing failures: 0


In [19]:
df["brand"] = df["name"].str.split().str[0]

print("Number of unique brands:", df["brand"].nunique())

df["brand"].value_counts().head(30)

Number of unique brands: 71


brand
Suzuki        19479
Toyota        17595
Honda         10917
Daihatsu       3424
Nissan         1741
Hyundai        1346
KIA            1235
Mitsubishi      834
Mercedes        589
Changan         552
MG              447
Haval           379
Audi            291
Mazda           194
BMW             193
FAW             190
Prince          153
DFSK            145
JAC             133
Lexus           120
Proton          112
Chevrolet        99
Peugeot          98
Subaru           89
Chery            83
Jeep             66
Daewoo           57
BYD              47
Porsche          44
Range            43
Name: count, dtype: int64

In [20]:
print("Total unique candidate brands:", df["brand"].nunique())

sorted(df["brand"].unique())

Total unique candidate brands: 71


['Adam',
 'Audi',
 'BAIC',
 'BAW',
 'BMW',
 'BYD',
 'Cadillac',
 'Changan',
 'Chery',
 'Chevrolet',
 'DFSK',
 'Daehan',
 'Daewoo',
 'Daihatsu',
 'Datsun',
 'Deepal',
 'Dodge',
 'Dongfeng',
 'FAW',
 'Fiat',
 'Ford',
 'GMC',
 'GUGO',
 'Genesis',
 'Haval',
 'Honda',
 'Honri',
 'Hummer',
 'Hyundai',
 'Infiniti',
 'Isuzu',
 'JAC',
 'JMC',
 'JW',
 'Jaguar',
 'Jeep',
 'Jetour',
 'KIA',
 'Kaiser',
 'Lamborghini',
 'Land',
 'Lexus',
 'MG',
 'MINI',
 'Master',
 'Mazda',
 'Mercedes',
 'Mitsubishi',
 'Mushtaq',
 'Nissan',
 'ORA',
 'Opel',
 'Peugeot',
 'Porsche',
 'Power',
 'Prince',
 'Proton',
 'Range',
 'Rinco',
 'Roma',
 'Seres',
 'Sogo',
 'Sokon',
 'SsangYong',
 'Subaru',
 'Suzuki',
 'Tesla',
 'Toyota',
 'United',
 'Volkswagen',
 'Volvo']

In [21]:
df.loc[
    df["brand"].isin(["Mercedes", "Land", "Range", "JW", "Power"]),
    "name"
].drop_duplicates().sort_values().head(100).tolist()

['JW Forland Bravo 1.0 2021',
 'JW Forland Bravo 2019',
 'JW Forland Bravo 2021',
 'JW Forland Bravo 3.0+ 2021',
 'JW Forland Bravo 3.0+ 2022',
 'JW Forland C-10  2025',
 'JW Forland Safari  2021',
 'JW Forland Safari  2025',
 'JW Forland Safari  Comfort 1.5 CL 2020',
 'JW Forland Safari  Deluxe 1.5 EFI 2024',
 'JW Forland T-5 2021',
 'Land Rover Defender 110 1995',
 'Land Rover Defender 110 2000',
 'Land Rover Defender 110 2004',
 'Land Rover Defender 110 2005',
 'Land Rover Defender 110 2006',
 'Land Rover Defender 110 2010',
 'Land Rover Defender 1969',
 'Land Rover Defender 2006',
 'Land Rover Defender 90 SW 1980',
 'Land Rover Discovery 1995',
 'Land Rover Discovery 1996',
 'Land Rover Freelander 1998',
 'Land Rover Freelander 2005',
 'Land Rover Freelander 2006',
 'Mercedes Benz 200 D 1996',
 'Mercedes Benz A Class 1998',
 'Mercedes Benz A Class 2002',
 'Mercedes Benz A Class 2004',
 'Mercedes Benz A Class 2005',
 'Mercedes Benz A Class 2019',
 'Mercedes Benz A Class A200  2020',

In [22]:
df.loc[
    df["brand"].isin(["Range", "Power"]),
    ["brand", "name"]
].drop_duplicates().sort_values(["brand", "name"]).to_string(index=False)

'brand                                       name\nPower                       Power  Mini Bus 2013\nPower                       Power  Mini Bus 2014\nRange             Range Rover Autobiography 2013\nRange             Range Rover Autobiography 2016\nRange       Range Rover Autobiography P400e 2018\nRange       Range Rover Autobiography P400e 2019\nRange       Range Rover Autobiography P400e 2020\nRange      Range Rover Evoque Autobiography 2010\nRange      Range Rover Evoque Autobiography 2021\nRange                     Range Rover Sport 2000\nRange                     Range Rover Sport 2016\nRange                     Range Rover Sport 2020\nRange              Range Rover Sport 5.0 V8 2010\nRange                 Range Rover Sport SVR 2016\nRange                 Range Rover Sport SVR 2020\nRange Range Rover Sport Supercharged 4.2 V8 2005\nRange Range Rover Sport Supercharged 4.2 V8 2006\nRange Range Rover Sport Supercharged 4.2 V8 2007\nRange                Range Rover Sport TDV6 2014\

In [23]:
brand_mapping = {
    "Mercedes": "Mercedes Benz",
    "Land": "Land Rover",
    "Range": "Range Rover",
    "JW": "JW Forland"
}

df["brand"] = df["brand"].replace(brand_mapping)

print("Unique brands after cleaning:", df["brand"].nunique())

df["brand"].value_counts().head(15)

Unique brands after cleaning: 71


brand
Suzuki           19479
Toyota           17595
Honda            10917
Daihatsu          3424
Nissan            1741
Hyundai           1346
KIA               1235
Mitsubishi         834
Mercedes Benz      589
Changan            552
MG                 447
Haval              379
Audi               291
Mazda              194
BMW                193
Name: count, dtype: int64

In [24]:
df = df.drop(
    columns=[
        "price",
        "mileage",
        "engine_capacity",
        "features"
    ]
)

print("Dataset shape:", df.shape)
print(df.columns.tolist())

Dataset shape: (61056, 14)
['name', 'year', 'fuel', 'transmission', 'registration_location', 'color', 'assembly', 'body_type', 'price_pkr', 'mileage_km', 'engine_cc', 'battery_kwh', 'feature_count', 'brand']


In [25]:
df[["brand", "name"]].sample(30, random_state=42)

,brand,name
2771,Nissan,Nissan Note 1.2E 2018
55398,Toyota,Toyota Corolla XLi VVTi 2015
59346,Toyota,Toyota Corolla XE 2001
33648,Suzuki,Suzuki Mehran VXR Euro II 2016
13422,Suzuki,Suzuki Swift DLX 1.3 2011
59058,Suzuki,Suzuki Alto VXR 2022
8055,Toyota,Toyota Corolla GLi 1.3 VVTi 2011
57269,Toyota,Toyota Corolla XLi VVTi 2013
4540,Honda,Honda Fit 1.5 Hybrid S Package 2014
36208,Suzuki,Suzuki Ciaz Automatic 2018


In [26]:
multi_word_brands = {
    "Mercedes Benz",
    "Land Rover",
    "Range Rover",
    "JW Forland"
}

def extract_model(row):
    parts = row["name"].split()

    if row["brand"] in multi_word_brands:
        return parts[2] if len(parts) > 2 else "Unknown"

    return parts[1] if len(parts) > 1 else "Unknown"


df["model"] = df.apply(extract_model, axis=1)

print("Unique models:", df["model"].nunique())
print("Unknown models:", (df["model"] == "Unknown").sum())

df["model"].value_counts().head(20)

Unique models: 460
Unknown models: 0


model
Corolla     7616
Civic       5152
Mehran      4638
Alto        4384
City        3480
Cultus      3467
Wagon       1566
Vitz        1458
Mira        1135
Bolan       1082
Swift       1056
Yaris        990
Raize        843
Passo        835
Prado        791
Cuore        763
Vezel        760
Prius        727
Sportage     702
Hilux        681
Name: count, dtype: int64

In [27]:
output_path = "../data/processed/cleaned_used_cars.csv"

df.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully.")
print("Final shape:", df.shape)
print("Saved to:", output_path)

Cleaned dataset saved successfully.
Final shape: (61056, 15)
Saved to: ../data/processed/cleaned_used_cars.csv
